题目 1：dataclass 建模
定义 Product，字段：sku: str、price: Decimal、stock: int；初始化时保证价格和库存非负。

In [2]:
from decimal import Decimal


class Product():
    def __init__(self, sku: str ,price: Decimal ,stock: int):
        if price <= 0 or stock <= 0:
            raise ValueError("Price and stock must be positive integers")
        self.sku = sku
        self.price = price
        self.stock = stock
        
product = Product("SKU-1", Decimal("19.90"), 5)
assert product.price == Decimal("19.90")

题目 2：业务逻辑与异常

实现 `InventoryService.reserve(sku, quantity)`，合法请求扣减库存；数量 <= 0、SKU 不存在、库存不足时分别抛出明确异常。要求业务操作集中在 Service 中。

In [3]:
class InventoryService():
    def __init__(self, stock: dict[str, int]) -> None:
        self._stock = stock.copy()
        

    def reserve(self, sku: str, quantity: int):
        if quantity <= 0:
            raise ValueError("Quantity must be a positive integer")
        elif self._stock.get(sku) is None:
            raise KeyError("SKU must not be None")
        elif self._stock[sku] < quantity:
            raise ValueError("Not enough stock available")
        
        self._stock[sku] -= quantity
        return self._stock[sku]

svc = InventoryService({"A": 5})
assert svc.reserve("A", 2) == 3

## 题目 3：Protocol 与依赖注入

编写 `UserRepository` 协议以及内存实现，并让 `UserService` 依赖协议而不是具体存储。

In [ ]:
from typing import Protocol

class UserRepository(Protocol):
    def find_name(self, user_id: int) -> str | None: ...

class InMemoryUserRepository:
    def __init__(self, users: dict[int, str]) -> None:
        self.users = users

    def find_name(self, user_id: int) -> str | None:
        return self.users.get(user_id)

class UserService:
    def __init__(self, repository: UserRepository) -> None:
        self.repository = repository

    def get_display_name(self, user_id: int) -> str:
        return self.repository.find_name(user_id) or "未知用户"
# InMemoryUserRepository在没有显示继承或实现UserRepository的情况下，仍然可以被UserService使用，
# 因为它符合UserRepository的接口要求（即具有find_name方法）。这就是鸭子类型的一个例子：只要对象具有所需的方法和属性，就可以被视为实现了该接口。
assert UserService(InMemoryUserRepository({1: "张三"})).get_display_name(1) == "张三"

In [4]:
from dataclasses import dataclass
from os import getenv
# 把环境变量读成一个不可变的、有类型标注的参数对象。
# frozen=True 让实例初始化后禁止重新赋值：settings.debug = True 会抛 FrozenInstanceError。
# 它带来两个实际好处：配置对象可以在模块间随便传，不会有人偷偷改掉；
# 类的 __hash__ 变成可用的，实例能放进 set、当作 dict 的键、被 lru_cache 缓存。
# 默认的 dataclass（非 frozen）会把 __hash__ 设为 None。代价是：frozen 时不能用继承再“加默认值”的方式覆盖字段，也不能有可变默认值（如 list、dict）——那个需要 field(default_factory=...)。
@dataclass(frozen=True)
class Settings:
    database_url: str
    debug: bool

    #@classmethod 把第一个参数从实例 self 换成类 cls。这里它当替代构造函数用（Java 里叫静态工厂）：Settings 本身的 __init__ 只接受显式值，方便测试里直接构造；真正读环境的逻辑放在 from_env，调用方写 Settings.from_env()，而不是 Settings.from_env(settings)。
    @classmethod
    #-> "Settings" 里加引号是因为方法体内定义时，Settings 这个类还没完全创建出来，直接写 Settings 在 Python 3.10 之前会 NameError。引号让它变成延迟求值的字符串（forward reference）。
    def from_env(cls) -> "Settings":
        #返回的是 cls(...) 而不是 Settings(...)，这样继承 Settings 的子类调用 from_env() 时，拿到的是子类实例——这是 classmethod 相对 staticmethod 的核心价值。
        return cls(
            database_url=getenv("DATABASE_URL", "sqlite:///local.db"),
            debug=getenv("DEBUG", "false").lower() == "true",
        )

In [ ]:
%pip install pytest

In [5]:
import pytest
from myapp.service import InventoryError, InventoryService

def test_reserve_success():
    svc = InventoryService({"A": 5})
    assert svc.reserve("A", 2) == 3

def test_insufficient_stock():
    svc = InventoryService({"A": 1})
    with pytest.raises(InventoryError, match="库存不足"):
        svc.reserve("A", 2)


ModuleNotFoundError: No module named 'pytest'